# Armar las tablas finales

Une los meses convertidos en `0.1` (todas las columnas como texto, tal como las publica la DIAN) en **una tabla por flujo y año**, con las columnas unificadas entre 2001 y hoy, sus tipos y algunas correcciones documentadas: `data/processed/importaciones_<año>.parquet` y `data/processed/exportaciones_<año>.parquet`.

El código está en `colombia_dian_comex/build.py`. Todas las decisiones salen de tres tablas versionadas en el repositorio, revisadas una por una, con la evidencia de cada una:

| Tabla | Qué define |
|---|---|
| `colombia_dian_comex/mapeo_columnas.csv` | Para cada columna original y rango de meses: a qué columna final va, o si se elimina, y por qué. |
| `colombia_dian_comex/esquema_final.csv` | Las columnas finales, en orden, con su tipo (texto, código, decimal, fecha) y si un "0" significa "no aplica". |
| `colombia_dian_comex/tablas/*.csv` | Nombres de códigos que la DIAN dejó como "no declarado o fuera de lista" y se recuperan. |

## Qué se hace con cada mes

1. **Unificar columnas** según `mapeo_columnas.csv`:
   - las columnas equivalentes que cambiaron de nombre se unen bajo el nombre actual de la DIAN (por ejemplo `IMPORTADOR` → `NOMBRE_IMPORTADOR`);
   - las columnas con el nombre corrido respecto de los datos (exportaciones 2004-2006) se mueven a la columna que corresponde a su contenido;
   - cuando una columna con el mismo nombre trajo otra cosa en años viejos, esos años van a una columna aparte con el período en el nombre (`NUMERO_SERIE_2001_2004`, `COD_MONEDA_TRANSACCION_2001_2003`, `COD_MONEDA_TRANSACCION_2004`, `COD_UNIDAD_COMERCIAL_2001_2002`);
   - las columnas que solo valen "1" en todos los meses se eliminan.

   Si un mes trae una columna que el mapeo no contempla, el proceso se detiene: una estructura nueva de la DIAN hay que revisarla antes de unificarla.
2. **Limpiar valores**:
   - se quitan los espacios al principio y al final, y el texto vacío queda vacío (nulo);
   - en los códigos se quita el `.0` que agrega Excel (`3.0` → `3`, `9.000000` → `9`);
   - la subpartida se completa a 10 dígitos con ceros a la izquierda (la DIAN publica algunas sin el cero inicial);
   - en las columnas de números y fechas de referencia donde la DIAN escribe `0` para decir "no aplica" (desde febrero de 2017; antes las dejaba vacías), el `0` pasa a vacío. Los montos en `0` quedan en `0`;
   - `IDENTIFICACION_FORMULARIO` queda vacía donde la DIAN la publicó en notación científica, que pierde dígitos (2004-10 a 2004-12 y 2008). El número completo está en `NUMERO_FORMULARIO`;
   - donde la DIAN escribió "no declarado o fuera de lista" pero el código es válido, se recupera el nombre del código: `TIPO_IMPORTACION` y `FORMA_PAGO` (2004 y 2008) y `TIPO_DESPACHO` de exportaciones 2004-2006 (a partir de `TIPO_DECLARACION`).
3. **Agregar columnas**: `PERIODO` (el mes del archivo, aaaamm) y, al lado de cada código de país, su código **M49** para todos los años (`<columna>_M49`). Desde 2025 la DIAN ya usa M49; hasta 2024 usaba ALADI y se convierte con la correlativa oficial de la DIAN (`docs/`). Los códigos sin equivalente M49 (`999` no declarados, Antillas Holandesas, Yugoslavia, zonas francas) quedan vacíos.
4. **Asignar tipos** según `esquema_final.csv`. Los códigos, NIT y números de documento quedan como texto, para no perder ceros iniciales; los montos, pesos y cantidades como decimales; las fechas aaaammdd como fecha. Los valores que no se pueden convertir (por ejemplo, fechas inválidas como `20010229` o `2008513`) quedan vacíos y se cuentan en el reporte de abajo.

## Controles

Para cada año se comprueba que la tabla final **sigue coincidiendo con los totales de control de la DIAN** (filas, FOB, CIF y peso de cada mes): unificar y tipar no puede perder ni duplicar filas ni montos. También se registra cuántos valores no se pudieron convertir a número o fecha y qué columnas tienen datos en cada año (`cobertura_columnas.parquet`, que usan las consultas para no mostrar columnas vacías en el período consultado).

In [ ]:
import logging

import polars as pl
from tqdm.auto import tqdm

from colombia_dian_comex import utils
from colombia_dian_comex.build import (
    build_year,
    interim_months,
    typing_losses,
    update_coverage,
    validate_year,
    year_coverage,
)
from colombia_dian_comex.pipeline import FIRST_YEAR, Flow

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

# Years with converted months, per flow.
years = {
    flow: [year for year in range(FIRST_YEAR, 2100) if interim_months(flow, year)]
    for flow in Flow
}

## Paso 1: armar, validar y medir cada año

Cada año se escribe en un archivo aparte; un año de importaciones tarda alrededor de un minuto.

In [ ]:
validations, losses, coverages = [], [], []
for flow, flow_years in years.items():
    for year in tqdm(flow_years, desc=f"Armando {flow}"):
        build_year(flow, year)
        validations.append(validate_year(flow, year))
        losses.append(typing_losses(flow, year))
        coverages.append(year_coverage(flow, year))

update_coverage(pl.concat(coverages))
df_validation = pl.concat(validations)
df_losses = pl.concat(losses)
df_validation.write_parquet(utils.data_interim_dir("validacion_tablas_finales.parquet"))
df_losses.write_parquet(utils.data_interim_dir("perdidas_tipado.parquet"))

## Paso 2: control contra los totales de la DIAN

Tiene que dar todo `OK`: abajo se muestran las comparaciones que no coinciden (debería estar vacío).

In [ ]:
logging.info(
    "%d de %d comparaciones coinciden con los totales de control.",
    df_validation["OK"].sum(),
    df_validation.height,
)
df_validation.filter(~pl.col("OK"))

## Paso 3: valores que no se pudieron convertir

Valores no vacíos (y distintos de `0`) que no son un número o una fecha válida, por columna y año. Quedan vacíos en la tabla final; el texto original sigue en `data/interim/`.

In [ ]:
(
    df_losses.group_by("FLUJO", "COLUMNA", "TIPO")
    .agg(
        pl.col("FILAS_PERDIDAS").sum(),
        pl.col("ANIO").min().alias("DESDE"),
        pl.col("ANIO").max().alias("HASTA"),
        pl.col("EJEMPLOS").first(),
    )
    .sort("FILAS_PERDIDAS", descending=True)
)